# Étape 5 – Machine Learning : prédire les avis négatifs

**Objectif métier :** repérer **à l'avance** les commandes qui risquent de recevoir un avis négatif (1 ou 2 étoiles),
pour que le service client puisse agir (prévenir le client, accélérer l'envoi, offrir un geste commercial…).

**Type de problème :** c'est une **classification binaire** — le modèle doit répondre *oui* (avis négatif) ou *non*.

**Le principe du Machine Learning supervisé :**
1. on montre au modèle des milliers de commandes **dont on connaît déjà la réponse** (la note) → c'est l'**entraînement** ;
2. le modèle apprend tout seul les liens entre les caractéristiques de la commande et la note ;
3. on vérifie ensuite s'il devine juste sur des commandes **qu'il n'a jamais vues** → c'est le **test**.

> 💡 Vocabulaire : les caractéristiques (délai, prix, distance…) s'appellent les **features** ;
> ce qu'on veut prédire (avis négatif oui/non) s'appelle la **cible** (*target*).

## 0. Préparation

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import pg_url

engine = create_engine(pg_url())
COULEUR = "#2a78d6"
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

## 1. Construire le jeu de données : une ligne par commande

On rassemble en **une seule requête SQL** tout ce qu'on sait sur chaque commande livrée :
prix, frais de port, poids et volume des produits, catégorie, États du client et du vendeur,
coordonnées GPS, moyen de paiement, délais… et bien sûr la note.

*(La requête est longue mais tu connais déjà toutes les briques : CTE, `JOIN`, `GROUP BY`, `LEFT JOIN`.
Deux nouveautés : `DISTINCT ON` garde une seule ligne par commande — l'article le plus cher —
et `ARRAY_AGG(...)[1]` récupère le moyen de paiement du plus gros paiement.)*

In [ ]:
REQUETE = """
    WITH notes AS (
        SELECT order_id, AVG(review_score) AS note
        FROM order_reviews
        GROUP BY order_id
    ),
    articles AS (
        SELECT i.order_id,
               COUNT(*)                                                         AS nb_articles,
               COUNT(DISTINCT i.seller_id)                                      AS nb_vendeurs,
               SUM(i.price)                                                     AS prix_total,
               SUM(i.freight_value)                                             AS frais_port,
               SUM(p.product_weight_g)                                          AS poids_total_g,
               SUM(p.product_length_cm * p.product_height_cm * p.product_width_cm) AS volume_total_cm3
        FROM order_items i
        JOIN products p ON p.product_id = i.product_id
        GROUP BY i.order_id
    ),
    article_principal AS (           -- l'article le plus cher de la commande
        SELECT DISTINCT ON (i.order_id)
               i.order_id, i.seller_id,
               COALESCE(t.product_category_name_english, 'inconnue') AS categorie
        FROM order_items i
        JOIN products p ON p.product_id = i.product_id
        LEFT JOIN category_translation t ON t.product_category_name = p.product_category_name
        ORDER BY i.order_id, i.price DESC
    ),
    paiements AS (
        SELECT order_id,
               MAX(payment_installments)                                AS nb_echeances,
               (ARRAY_AGG(payment_type ORDER BY payment_value DESC))[1] AS moyen_paiement
        FROM order_payments
        GROUP BY order_id
    )
    SELECT o.order_id,
           EXTRACT(MONTH FROM o.order_purchase_timestamp)::int                            AS mois_achat,
           o.order_estimated_delivery_date::date - o.order_purchase_timestamp::date       AS delai_promis_jours,
           a.nb_articles, a.nb_vendeurs, a.prix_total, a.frais_port,
           a.poids_total_g, a.volume_total_cm3,
           ap.categorie,
           c.customer_state AS etat_client,
           s.seller_state   AS etat_vendeur,
           gc.lat AS lat_client,  gc.lng AS lng_client,
           gs.lat AS lat_vendeur, gs.lng AS lng_vendeur,
           pm.moyen_paiement, pm.nb_echeances,
           o.order_delivered_customer_date::date - o.order_purchase_timestamp::date      AS delai_jours,
           o.order_delivered_customer_date::date - o.order_estimated_delivery_date::date AS ecart_jours,
           n.note
    FROM orders o
    JOIN notes             n  ON n.order_id  = o.order_id
    JOIN articles          a  ON a.order_id  = o.order_id
    JOIN article_principal ap ON ap.order_id = o.order_id
    JOIN customers         c  ON c.customer_id = o.customer_id
    JOIN sellers           s  ON s.seller_id   = ap.seller_id
    LEFT JOIN geolocation_zip gc ON gc.zip_code_prefix = c.customer_zip_code_prefix
    LEFT JOIN geolocation_zip gs ON gs.zip_code_prefix = s.seller_zip_code_prefix
    LEFT JOIN paiements       pm ON pm.order_id = o.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
"""

df = pd.read_sql(REQUETE, engine)
print(f"{len(df):,} commandes x {df.shape[1]} colonnes".replace(",", " "))
df.head()

## 2. Créer de nouvelles features (« feature engineering »)

Les données brutes ne suffisent pas toujours : on **fabrique** des variables plus parlantes pour le modèle.
- **`distance_km`** : distance à vol d'oiseau entre le client et le vendeur (formule de **haversine**, qui tient compte de la courbure de la Terre)
- **`meme_etat`** : le vendeur est-il dans le même État que le client ? (1 = oui, 0 = non)
- **`avis_negatif`** : notre **cible** → 1 si la note est ≤ 2, sinon 0

In [ ]:
def haversine_km(lat1, lng1, lat2, lng2):
    """Distance en km entre deux points GPS."""
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine_km(df["lat_client"], df["lng_client"], df["lat_vendeur"], df["lng_vendeur"])
df["meme_etat"] = (df["etat_client"] == df["etat_vendeur"]).astype(int)
df["avis_negatif"] = (df["note"] <= 2).astype(int)

df[["distance_km", "meme_etat", "avis_negatif"]].describe().round(2)

## 3. La cible est-elle équilibrée ?

In [ ]:
repartition = df["avis_negatif"].value_counts(normalize=True).mul(100).round(1)
repartition.index = ["Avis positif / neutre (0)", "Avis négatif (1)"]
repartition

➡️ Les classes sont **déséquilibrées** : il y a beaucoup plus de clients contents que de mécontents.
On verra que c'est un **piège** pour évaluer un modèle.

## 4. ⚠️ La fuite de données (« data leakage »)

C'est **l'erreur n°1** en Machine Learning, et une question classique d'entretien.

Imagine qu'on donne au modèle `ecart_jours` (le retard réel). Il deviendrait très bon… mais c'est de la **triche** :
**au moment où l'on veut prédire** (juste après l'achat), le colis n'est pas encore livré, on **ne connaît pas encore** le retard !
Un modèle qui utilise une information du futur paraît excellent en test… et ne sert à rien en vrai.

On construit donc **deux scénarios** :

| Scénario | Quand prédit-on ? | Features autorisées |
|---|---|---|
| **A – À l'achat** | juste après la commande | tout ce qu'on sait à l'achat (prix, distance, délai promis…) |
| **B – À la livraison** | quand le colis arrive, **avant** que le client note | A + le délai réel et le retard |

Le scénario B est utile aussi : le service client peut contacter le client **entre la livraison et l'avis**.

In [ ]:
FEATURES_NUM_ACHAT = ["delai_promis_jours", "nb_articles", "nb_vendeurs", "prix_total", "frais_port",
                      "poids_total_g", "volume_total_cm3", "distance_km", "meme_etat",
                      "nb_echeances", "mois_achat"]
FEATURES_CAT = ["categorie", "etat_client", "moyen_paiement"]
FEATURES_LIVRAISON = ["delai_jours", "ecart_jours"]

FEATURES_A = FEATURES_NUM_ACHAT + FEATURES_CAT                       # scénario A
FEATURES_B = FEATURES_NUM_ACHAT + FEATURES_LIVRAISON + FEATURES_CAT  # scénario B

print(f"Scénario A : {len(FEATURES_A)} features")
print(f"Scénario B : {len(FEATURES_B)} features")

## 5. Séparer entraînement et test

On coupe les données en deux :
- **80 % pour l'entraînement** : le modèle apprend sur ces commandes ;
- **20 % pour le test** : on les cache au modèle, puis on vérifie ses prédictions dessus.

C'est comme un examen : si on donnait à l'élève les questions de l'examen pendant ses révisions,
sa note ne mesurerait pas ce qu'il a vraiment compris.

`stratify=y` garde **la même proportion d'avis négatifs** dans les deux parties.
`random_state=42` rend le découpage **reproductible** (même résultat à chaque exécution).

In [ ]:
from sklearn.model_selection import train_test_split

X = df[FEATURES_B]          # on garde toutes les colonnes ; chaque modèle choisira les siennes
y = df["avis_negatif"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Entraînement : {len(X_train):,} commandes".replace(",", " "))
print(f"Test         : {len(X_test):,} commandes".replace(",", " "))
print(f"% d'avis négatifs : train {100 * y_train.mean():.1f} % | test {100 * y_test.mean():.1f} %")

## 6. Le modèle « bête » de référence (baseline)

Avant de construire un vrai modèle, on se demande : **quel score obtiendrait un modèle qui ne réfléchit pas ?**

Le plus simple : répondre **toujours « avis positif »**. On mesure deux choses :
- **Accuracy (exactitude)** : % de prédictions correctes, toutes classes confondues ;
- **Rappel (recall) des avis négatifs** : parmi les **vrais** avis négatifs, combien le modèle en a-t-il **détectés** ?

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)

print(f"Accuracy                   : {100 * accuracy_score(y_test, pred):.1f} %")
print(f"Rappel des avis négatifs   : {100 * recall_score(y_test, pred):.1f} %")

### 🎯 À toi d'interpréter

1. L'accuracy du modèle « bête » est-elle **bonne ou mauvaise** à première vue ? Pourquoi est-elle si élevée ?
2. Que vaut son **rappel** ? Ce modèle est-il **utile** pour le service client ?
3. Si un collègue t'annonce fièrement « mon modèle a 85 % d'accuracy ! », que lui réponds-tu ?
4. Pour chacune de ces features, dis si elle est autorisée dans le **scénario A** (à l'achat) : `frais_port`, `ecart_jours`, `distance_km`, `note`.

✍️ *Écris tes réponses ici :*

…